# GISPR Module 3 — Vector Data in Python and R

**Course:** GIS Spatial Analysis with Python and R (GISPR)  
**Module:** 3 — Vector Data: Formats, Geometry, CRS, and Attributes  
**Builds on:** Module 2 (loading spatial data, loops, functions, Git basics)

---

### What you'll accomplish this module

By the end of this notebook you'll be able to:

| # | Learning outcome | Where |
|---|---|---|
| 1 | Read and write the common vector formats (Shapefile, GeoJSON, GeoPackage, File GDB) in both languages | Section 1 |
| 2 | Construct and inspect Point, LineString, and Polygon geometries with Shapely and `sf` | Section 2 |
| 3 | Understand and transform CRS — when to reproject and how to do it correctly | Section 3 |
| 4 | Manipulate attribute tables: add, rename, retype, filter, and compute derived columns | Section 4 |
| 5 | Produce a quick static map of your vector data and export a clean GeoPackage | Section 5 |
| 6 | Access vector data via ArcPy and the ArcGIS API for Python | Section 6 |

### Kernel reminder
- 🔵 **`[R]`** cells — switch kernel to **R** before running
- 🟢 **`[Python]`** cells — switch kernel to **Python 3** (or your cloned ArcGIS Pro env) before running
- 📋 **`[Terminal]`** cells — paste the command into your terminal / Anaconda Prompt, **not** in a notebook cell

> **The framing problem for this module:**  
> Your agency receives a GIS data request every Monday: "attach 2023 permit counts by census tract."  
> The permits arrive as a CSV. The tracts live in a File GDB. The final product must be a GeoPackage for web upload.  
> That's three formats, one join, and a CRS check — all of which you'll be able to script by the end of this notebook.

---
## Section 1 — Reading and Writing Vector Formats

The GIS world has accumulated dozens of vector formats over 30 years. Knowing how to read and write the most common ones — and understanding their tradeoffs — is foundational before any analysis begins.

### Format quick-reference

| Format | Extension | Strengths | Watch out for |
|---|---|---|---|
| Shapefile | `.shp` + 3–6 siblings | Universal, ESRI-native | 10-char field names, 2 GB limit, no datetime |
| GeoJSON | `.geojson` | Human-readable, web-native | Large files, WGS 84 only by convention |
| GeoPackage | `.gpkg` | Modern, single-file, multi-layer | Slightly less universal |
| File Geodatabase | `.gdb` | ESRI standard, domains, subtypes | Proprietary — read/write via `arcpy`, read-only via GDAL |
| CSV (with X/Y cols) | `.csv` | Easy to share tabular point data | No geometry type declared; CRS must be inferred |

Both `geopandas` (Python) and `sf` (R) sit on top of **GDAL/OGR**, so the same driver handles all of these — the API call is nearly identical regardless of format.

### 1a — Read multiple formats in Python with `geopandas`

In [ ]:
# [Python] Reading vector formats with geopandas
# All use gpd.read_file() — GDAL handles the driver selection automatically

import geopandas as gpd
import pandas as pd
from geodatasets import get_path   # pip install geodatasets — bundled sample data

# ── Example 1: Read a Shapefile ───────────────────────────────────────────────
# Using geodatasets for a reproducible, download-free example
shp_path = get_path('naturalearth.land')   # returns path to a local .shp file
land = gpd.read_file(shp_path)
print("Shapefile loaded:", type(land))
print("  Rows:", len(land), "  CRS:", land.crs)

# ── Example 2: Read GeoJSON ────────────────────────────────────────────────────
# Also works with a URL — useful for live web data
geojson_url = (
    "https://raw.githubusercontent.com/python-visualization/folium"
    "/main/examples/data/us-states.json"
)
# Uncomment if you have network access in your environment:
# states_gj = gpd.read_file(geojson_url)

# ── Example 3: Read a GeoPackage (single or multi-layer) ─────────────────────
# gpkg_path = 'data/my_project.gpkg'
# parcels = gpd.read_file(gpkg_path, layer='parcels')   # specify layer by name

# List layers in a GeoPackage without loading them:
# import fiona
# print(fiona.listlayers(gpkg_path))

# ── Example 4: Read a CSV with X/Y columns → GeoDataFrame ─────────────────────
# This is the most common pattern for point data (monitoring stations, permits, etc.)
csv_data = {
    'station_id': ['S01', 'S02', 'S03', 'S04'],
    'name':       ['Sea-Tac', 'Olympia', 'Bellingham', 'Spokane'],
    'lon':        [-122.3088, -122.9007, -122.5403, -117.4260],
    'lat':        [  47.4444,   47.0379,   48.7519,   47.6588],
    'precip_cm':  [    96.0,    123.7,      95.3,      43.0]
}
df = pd.DataFrame(csv_data)

# Convert to GeoDataFrame by specifying the geometry columns and CRS
stations = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['lon'], df['lat']),
    crs='EPSG:4326'   # WGS 84 geographic — the right CRS for lat/lon degree columns
)
print("\nCSV → GeoDataFrame:")
print(stations[['station_id', 'name', 'precip_cm', 'geometry']].head())
print("  Geometry type:", stations.geom_type.unique())
print("  CRS:", stations.crs)

### 1b — Read multiple formats in R with `sf`

In [ ]:
# [R] Reading vector formats with sf
# All use st_read() — driver selected automatically from file extension

library(sf)
library(spData)   # install.packages('spData') if needed

# ── Example 1: Built-in sf dataset (polygon) ──────────────────────────────────
world <- world   # sf object from spData — 177 country polygons
cat("Loaded sf object with", nrow(world), "features\n")
cat("CRS:", st_crs(world)$Name, "\n")

# ── Example 2: Read a Shapefile ───────────────────────────────────────────────
# shp_path <- 'data/parcels.shp'
# parcels <- st_read(shp_path, quiet = TRUE)  # quiet = TRUE suppresses verbose output

# ── Example 3: Read a GeoPackage layer ───────────────────────────────────────
# gpkg_path <- 'data/my_project.gpkg'
# List layers first:
# st_layers(gpkg_path)
# Read a specific layer:
# roads <- st_read(gpkg_path, layer = 'roads', quiet = TRUE)

# ── Example 4: Read a CSV with X/Y columns → sf ───────────────────────────────
stations_df <- data.frame(
    station_id = c('S01', 'S02', 'S03', 'S04'),
    name       = c('Sea-Tac', 'Olympia', 'Bellingham', 'Spokane'),
    lon        = c(-122.3088, -122.9007, -122.5403, -117.4260),
    lat        = c(  47.4444,   47.0379,   48.7519,   47.6588),
    precip_cm  = c(    96.0,    123.7,      95.3,      43.0)
)

# st_as_sf() promotes a plain data frame to an sf object
stations_sf <- st_as_sf(
    stations_df,
    coords = c('lon', 'lat'),   # column names for X (longitude) then Y (latitude)
    crs    = 4326               # EPSG code — WGS 84 geographic
)

cat("\nCSV → sf data frame:\n")
print(stations_sf)
cat("\nGeometry type:", as.character(unique(st_geometry_type(stations_sf))), "\n")

### 1c — Writing vector data to disk

In [ ]:
# [Python] Writing GeoDataFrames — GeoPackage is the preferred output format
import geopandas as gpd
import pandas as pd

# Rebuild stations GeoDataFrame from previous cell
csv_data = {
    'station_id': ['S01', 'S02', 'S03', 'S04'],
    'name':       ['Sea-Tac', 'Olympia', 'Bellingham', 'Spokane'],
    'lon':        [-122.3088, -122.9007, -122.5403, -117.4260],
    'lat':        [  47.4444,   47.0379,   48.7519,   47.6588],
    'precip_cm':  [    96.0,    123.7,      95.3,      43.0]
}
df = pd.DataFrame(csv_data)
stations = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['lon'], df['lat']),
    crs='EPSG:4326'
)

# ── Write GeoPackage (preferred: single file, no 10-char field limit) ─────────
stations.to_file('output/stations.gpkg', driver='GPKG', layer='wx_stations')
print("Wrote GeoPackage: output/stations.gpkg")

# ── Write GeoJSON (good for web sharing) ─────────────────────────────────────
stations.to_file('output/stations.geojson', driver='GeoJSON')
print("Wrote GeoJSON:    output/stations.geojson")

# ── Write Shapefile (for ArcGIS compatibility) ────────────────────────────────
# Warning: field names truncated to 10 chars; datetime loses timezone
stations.to_file('output/stations.shp')
print("Wrote Shapefile:  output/stations.shp")

# ── Add a second layer to an existing GeoPackage ─────────────────────────────
# mode='a' appends; use a different layer name
# boundary.to_file('output/stations.gpkg', driver='GPKG', layer='study_area', mode='a')

In [ ]:
# [R] Writing sf objects
library(sf)

# Rebuild stations_sf from previous R cell
stations_df <- data.frame(
    station_id = c('S01', 'S02', 'S03', 'S04'),
    name       = c('Sea-Tac', 'Olympia', 'Bellingham', 'Spokane'),
    lon        = c(-122.3088, -122.9007, -122.5403, -117.4260),
    lat        = c(  47.4444,   47.0379,   48.7519,   47.6588),
    precip_cm  = c(    96.0,    123.7,      95.3,      43.0)
)
stations_sf <- st_as_sf(stations_df, coords = c('lon', 'lat'), crs = 4326)

# ── Write GeoPackage ─────────────────────────────────────────────────────────
st_write(stations_sf, 'output/stations_r.gpkg', layer = 'wx_stations',
         delete_layer = TRUE, quiet = TRUE)   # delete_layer = TRUE overwrites if layer exists
cat("Wrote: output/stations_r.gpkg\n")

# ── Write GeoJSON ────────────────────────────────────────────────────────────
st_write(stations_sf, 'output/stations_r.geojson', delete_dsn = TRUE, quiet = TRUE)
cat("Wrote: output/stations_r.geojson\n")

# ── Write Shapefile ──────────────────────────────────────────────────────────
st_write(stations_sf, 'output/stations_r.shp', delete_layer = TRUE, quiet = TRUE)
cat("Wrote: output/stations_r.shp\n")

# ── Append a second layer to the same GeoPackage ─────────────────────────────
# st_write(boundary_sf, 'output/stations_r.gpkg', layer = 'study_area', append = TRUE)

---
## Section 2 — Geometry Types: Points, Lines, and Polygons

Every vector feature is one of three fundamental geometry types, or a "Multi" variant holding a collection of them:

| Type | Multi variant | Typical use case |
|---|---|---|
| Point | MultiPoint | Monitoring stations, incidents, address locations |
| LineString | MultiLineString | Roads, streams, pipelines, contours |
| Polygon | MultiPolygon | Census tracts, parcels, watersheds, countries |

In Python, **Shapely** constructs individual geometries. GeoPandas stores them in the `geometry` column.  
In R, **sf** handles geometry construction and storage together via `st_point()`, `st_linestring()`, `st_polygon()` and friends.

> **Why does this matter to me?** Some geoprocessing tools only accept specific geometry types. Knowing how to check, cast, and construct geometries in code means you can prepare data programmatically instead of using "Edit Geometry" in ArcGIS Pro.

### 2a — Construct all three geometry types in Python (Shapely)

In [ ]:
# [Python] Constructing Point, LineString, and Polygon geometries with Shapely

import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString, Polygon, MultiPolygon

# ── Point ─────────────────────────────────────────────────────────────────────
space_needle = Point(-122.3493, 47.6205)   # (longitude, latitude) — always X, Y
print(f"Point:          {space_needle}")
print(f"  Type:         {space_needle.geom_type}")
print(f"  Coordinates:  x={space_needle.x:.4f}, y={space_needle.y:.4f}")
print(f"  Is valid:     {space_needle.is_valid}")

# ── LineString ────────────────────────────────────────────────────────────────
# A simplified route segment — list of (lon, lat) tuples
aurora_segment = LineString([
    (-122.3440, 47.6900),   # north end
    (-122.3420, 47.6600),
    (-122.3400, 47.6300),   # south end
])
print(f"\nLineString:     {aurora_segment.wkt[:60]}...")
print(f"  Type:         {aurora_segment.geom_type}")
print(f"  Length:       {aurora_segment.length:.6f} degrees")
print(f"  Num points:   {len(aurora_segment.coords)}")

# ── Polygon ──────────────────────────────────────────────────────────────────
# A simplified bounding box around Green Lake, Seattle
# Exterior ring: list of (lon, lat) tuples — first and last must match
green_lake_approx = Polygon([
    (-122.3395, 47.6823),
    (-122.3295, 47.6823),
    (-122.3295, 47.6763),
    (-122.3395, 47.6763),
    (-122.3395, 47.6823),   # close the ring
])
print(f"\nPolygon:        {green_lake_approx.wkt[:60]}...")
print(f"  Type:         {green_lake_approx.geom_type}")
print(f"  Area:         {green_lake_approx.area:.8f} sq degrees")
print(f"  Perimeter:    {green_lake_approx.length:.6f} degrees")
print(f"  Is valid:     {green_lake_approx.is_valid}")

# ── Geometric relationships between objects ───────────────────────────────────
print(f"\nSpace Needle inside Green Lake polygon? {green_lake_approx.contains(space_needle)}")
print(f"Distance (degrees) Needle → Lake:       {green_lake_approx.distance(space_needle):.6f}")

# ── Package into a GeoDataFrame for mapping ────────────────────────────────────
geometries = gpd.GeoDataFrame({
    'name':    ['Space Needle', 'Aurora Segment', 'Green Lake Approx'],
    'type':    ['Point', 'LineString', 'Polygon'],
    'geometry': [space_needle, aurora_segment, green_lake_approx]
}, crs='EPSG:4326')

# Quick visual
fig, ax = plt.subplots(figsize=(6, 6))
geometries[geometries.geom_type == 'Polygon'].plot(ax=ax, color='#b7a57a', alpha=0.4, edgecolor='#4b2e83', linewidth=2)
geometries[geometries.geom_type == 'LineString'].plot(ax=ax, color='#4b2e83', linewidth=2)
geometries[geometries.geom_type == 'Point'].plot(ax=ax, color='#4b2e83', markersize=80, zorder=5)
ax.set_title('Point · LineString · Polygon — Seattle', fontsize=12)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()

### 2b — Construct all three geometry types in R (`sf`)

In [ ]:
# [R] Constructing Point, LineString, and Polygon geometries with sf

library(sf)

# ── Point ─────────────────────────────────────────────────────────────────────
# st_point() takes a numeric vector c(x, y) — x = longitude, y = latitude
space_needle_pt  <- st_point(c(-122.3493, 47.6205))
cat("Point:        ", class(space_needle_pt), "\n")
cat("  WKT:        ", st_as_text(space_needle_pt), "\n")
cat("  Coordinates:", space_needle_pt[[1]], space_needle_pt[[2]], "\n")

# ── LineString ────────────────────────────────────────────────────────────────
# st_linestring() takes a 2-column matrix of (x, y) vertices
aurora_pts <- rbind(
    c(-122.3440, 47.6900),
    c(-122.3420, 47.6600),
    c(-122.3400, 47.6300)
)
aurora_ls <- st_linestring(aurora_pts)
cat("\nLineString:   ", class(aurora_ls), "\n")
cat("  Length (deg):", round(st_length(st_sfc(aurora_ls)), 6), "\n")

# ── Polygon ──────────────────────────────────────────────────────────────────
# st_polygon() takes a LIST of rings — first ring is exterior, additional rings are holes
gl_ring <- rbind(
    c(-122.3395, 47.6823),
    c(-122.3295, 47.6823),
    c(-122.3295, 47.6763),
    c(-122.3395, 47.6763),
    c(-122.3395, 47.6823)   # close the ring — first == last
)
green_lake_poly <- st_polygon(list(gl_ring))
cat("\nPolygon:      ", class(green_lake_poly), "\n")
cat("  Area (sq deg):", round(st_area(st_sfc(green_lake_poly)), 8), "\n")
cat("  Is valid:     ", st_is_valid(st_sfc(green_lake_poly)), "\n")

# ── Spatial relationships ─────────────────────────────────────────────────────
cat("\nNeedle inside Green Lake polygon?",
    st_within(st_sfc(space_needle_pt, crs = 4326),
              st_sfc(green_lake_poly, crs = 4326),
              sparse = FALSE)[1,1], "\n")

# ── Package into an sf object ─────────────────────────────────────────────────
geom_list <- st_sfc(
    space_needle_pt,
    aurora_ls,
    green_lake_poly,
    crs = 4326
)

features_sf <- st_sf(
    name = c('Space Needle', 'Aurora Segment', 'Green Lake Approx'),
    type = c('Point', 'LineString', 'Polygon'),
    geometry = geom_list
)

print(features_sf)
plot(st_geometry(features_sf),
     col = c('#4b2e83', '#4b2e83', '#b7a57a'),
     main = 'Point · LineString · Polygon — Seattle',
     axes = TRUE)

### 2c — Checking and repairing geometry validity

Invalid geometries (self-intersecting rings, duplicate vertices, unclosed rings) will silently break spatial operations — or throw cryptic errors. Always validate before analysis.

In [ ]:
# [Python] Checking and fixing geometry validity

import geopandas as gpd
from shapely.geometry import Polygon
from shapely.validation import explain_validity

# Create a self-intersecting polygon (a 'bowtie' — invalid)
bowtie = Polygon([(0, 0), (2, 2), (2, 0), (0, 2), (0, 0)])

print("Is valid:", bowtie.is_valid)
print("Reason:  ", explain_validity(bowtie))

# Fix with buffer(0) — the standard geometry repair trick
fixed = bowtie.buffer(0)
print("\nAfter buffer(0):")
print("  Is valid:", fixed.is_valid)
print("  Type:    ", fixed.geom_type)  # may become MultiPolygon

# ── In a GeoDataFrame: check and fix the whole column ─────────────────────────
# gdf['is_valid'] = gdf.geometry.is_valid
# invalid = gdf[~gdf['is_valid']]
# print(f"{len(invalid)} invalid geometries found")

# Fix all at once:
# gdf.geometry = gdf.geometry.buffer(0)   # quick fix — check results carefully
# Or use make_valid (shapely >= 1.8):
# from shapely.ops import make_valid
# gdf.geometry = gdf.geometry.apply(make_valid)

In [ ]:
# [R] Checking and fixing geometry validity with sf

library(sf)

# Build the same bowtie
bowtie_ring <- rbind(c(0,0), c(2,2), c(2,0), c(0,2), c(0,0))
bowtie_poly <- st_polygon(list(bowtie_ring))
bowtie_sfc  <- st_sfc(bowtie_poly)

cat("Is valid:", st_is_valid(bowtie_sfc), "\n")
cat("Reason:  ", st_is_valid(bowtie_sfc, reason = TRUE), "\n")

# Fix with st_make_valid() — wraps lwgeom::lwgeom_make_valid()
# install.packages('lwgeom') if needed
# fixed_sfc <- st_make_valid(bowtie_sfc)
# cat("After st_make_valid — is valid:", st_is_valid(fixed_sfc), "\n")

# ── In an sf data frame: check the geometry column ───────────────────────────
# check_result <- st_is_valid(my_sf)
# cat(sum(!check_result), "invalid geometries\n")
# my_sf_fixed <- st_make_valid(my_sf)

---
## Section 3 — Coordinate Reference Systems: Understanding, Checking, and Reprojecting

CRS errors are the #1 source of silent failures in spatial analysis. A layer that plots in the wrong place, measurements in degrees instead of meters, or an overlay that's thousands of kilometers off — all CRS problems.

### The four CRS questions to ask every time you load data

1. **What CRS is it?** — `gdf.crs` / `st_crs(sf_obj)`
2. **Is it geographic or projected?** — degrees (geographic) vs. meters/feet (projected)
3. **Does it match my other layers?** — they must share a CRS before any spatial operation
4. **Is it appropriate for my analysis?** — distance/area measurements require a projected CRS in appropriate units

### Common EPSG codes for Pacific Northwest work

| EPSG | Name | Type | Units | Use case |
|---|---|---|---|---|
| 4326 | WGS 84 | Geographic | Degrees | GPS, web data, GeoJSON |
| 3857 | Web Mercator | Projected | Meters | Web tile basemaps (not for measurement) |
| 2927 | WA State Plane South (ft) | Projected | US survey feet | WA state/county data |
| 32610 | UTM Zone 10N (WGS 84) | Projected | Meters | Distance/area measurement in W WA |
| 4269 | NAD 83 | Geographic | Degrees | USGS and federal data |

> **`set_crs` vs. `to_crs` (Python) / `st_set_crs` vs. `st_transform` (R):**  
> - **Set** = "I'm declaring what the CRS already is" — use only when CRS is missing or wrong in metadata. Does not move coordinates.  
> - **Transform/reproject** = "Convert the coordinates to a new CRS" — use to align layers. Moves coordinates.

### 3a — CRS inspection and reprojection in Python

In [ ]:
# [Python] CRS inspection, assignment, and reprojection

import geopandas as gpd
import pandas as pd

# Build our Washington stations GeoDataFrame
df = pd.DataFrame({
    'station_id': ['S01', 'S02', 'S03', 'S04'],
    'name':       ['Sea-Tac', 'Olympia', 'Bellingham', 'Spokane'],
    'lon':        [-122.3088, -122.9007, -122.5403, -117.4260],
    'lat':        [  47.4444,   47.0379,   48.7519,   47.6588],
    'precip_cm':  [    96.0,    123.7,      95.3,      43.0]
})
stations_4326 = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['lon'], df['lat']),
    crs='EPSG:4326'
)

# ── Inspect CRS ───────────────────────────────────────────────────────────────
print("CRS:",              stations_4326.crs)
print("EPSG code:",        stations_4326.crs.to_epsg())
print("Is geographic?:",   stations_4326.crs.is_geographic)
print("Is projected?:",    stations_4326.crs.is_projected)
print("Linear units:",     stations_4326.crs.axis_info[0].unit_name)

# Sample coordinate — degrees
print("\nSample coord (WGS 84):", stations_4326.geometry.iloc[0])

# ── Reproject to UTM Zone 10N (EPSG:32610) for distance measurement ──────────
stations_utm = stations_4326.to_crs(epsg=32610)
print("\nAfter reproject to UTM 10N:")
print("  CRS:", stations_utm.crs.name)
print("  Linear units:", stations_utm.crs.axis_info[0].unit_name)
print("  Sample coord (meters):", stations_utm.geometry.iloc[0])

# ── Compute distances now that we're in a projected CRS ──────────────────────
seatac = stations_utm[stations_utm.station_id == 'S01'].geometry.iloc[0]
spokane = stations_utm[stations_utm.station_id == 'S04'].geometry.iloc[0]
dist_m = seatac.distance(spokane)
print(f"\nSea-Tac to Spokane: {dist_m/1000:.1f} km  ({dist_m/1609.34:.1f} miles)")

# ── Reproject to WA State Plane South (EPSG:2927, US survey feet) ─────────────
stations_wasp = stations_4326.to_crs(epsg=2927)
print("\nWA State Plane South units:", stations_wasp.crs.axis_info[0].unit_name)

# ── Assigning CRS when it's missing (use with caution) ────────────────────────
# Only do this if the dataset truly has no CRS assigned and you know what it should be
# stations_no_crs = stations_4326.set_crs(None)  # strip CRS for demo
# stations_reassigned = stations_no_crs.set_crs('EPSG:4326')  # assign — no coord change
# stations_reassigned_utm = stations_reassigned.to_crs('EPSG:32610')  # now reproject

### 3b — CRS inspection and reprojection in R

In [ ]:
# [R] CRS inspection, assignment, and reprojection with sf

library(sf)

# Build stations sf object
stations_df <- data.frame(
    station_id = c('S01', 'S02', 'S03', 'S04'),
    name       = c('Sea-Tac', 'Olympia', 'Bellingham', 'Spokane'),
    lon        = c(-122.3088, -122.9007, -122.5403, -117.4260),
    lat        = c(  47.4444,   47.0379,   48.7519,   47.6588),
    precip_cm  = c(    96.0,    123.7,      95.3,      43.0)
)
stations_4326 <- st_as_sf(stations_df, coords = c('lon', 'lat'), crs = 4326)

# ── Inspect CRS ───────────────────────────────────────────────────────────────
cat("CRS name:    ", st_crs(stations_4326)$Name, "\n")
cat("EPSG code:   ", st_crs(stations_4326)$epsg, "\n")
cat("Is geographic:", st_is_longlat(stations_4326), "\n")  # TRUE = geographic
cat("\nFull CRS info:\n")
print(st_crs(stations_4326))

# Sample coordinate in degrees
cat("\nSample coord (WGS 84):\n")
print(st_geometry(stations_4326)[[1]])

# ── Reproject to UTM Zone 10N ─────────────────────────────────────────────────
stations_utm <- st_transform(stations_4326, crs = 32610)
cat("\nAfter reproject to UTM 10N:\n")
cat("  CRS:   ", st_crs(stations_utm)$Name, "\n")
cat("  Sample coord (meters):\n")
print(st_geometry(stations_utm)[[1]])

# ── Compute distances in a projected CRS ──────────────────────────────────────
seatac  <- stations_utm[stations_utm$station_id == 'S01', ]
spokane <- stations_utm[stations_utm$station_id == 'S04', ]
dist_m  <- as.numeric(st_distance(seatac, spokane))
cat("\nSea-Tac to Spokane:", round(dist_m/1000, 1), "km",
    "(", round(dist_m/1609.34, 1), "miles)\n")

# ── Set CRS when missing (use with caution — does not transform) ──────────────
# stations_no_crs <- st_set_crs(stations_4326, NA)          # strip CRS for demo
# stations_reset  <- st_set_crs(stations_no_crs, 4326)      # assign — no coord change
# stations_utm2   <- st_transform(stations_reset, crs = 32610)

### 3c — CRS alignment check before overlay operations

This is a pattern you'll use in every multi-layer workflow. Always confirm CRS match before any spatial operation.

In [ ]:
# [Python] Defensive CRS check function — use at the top of any multi-layer workflow

import geopandas as gpd
import pandas as pd

def ensure_same_crs(*gdfs, target_epsg=32610):
    """
    Verify all GeoDataFrames share a CRS.
    If they differ, reproject all to target_epsg.
    Returns a tuple of aligned GeoDataFrames.
    """
    crs_values = [gdf.crs.to_epsg() if gdf.crs else None for gdf in gdfs]
    unique_crs = set(crs_values)

    if len(unique_crs) == 1 and None not in unique_crs:
        print(f"✅  All layers share CRS EPSG:{list(unique_crs)[0]}")
        return gdfs
    else:
        print(f"⚠️  CRS mismatch detected: {crs_values}")
        print(f"    Reprojecting all to EPSG:{target_epsg}...")
        reprojected = tuple(gdf.to_crs(epsg=target_epsg) for gdf in gdfs)
        print(f"✅  All layers now in EPSG:{target_epsg}")
        return reprojected

# --- Demo ---------------------------------------------------------------
df = pd.DataFrame({
    'station_id': ['S01', 'S02'],
    'lon': [-122.3088, -122.9007],
    'lat': [  47.4444,   47.0379]
})
layer_a = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs='EPSG:4326')
layer_b = layer_a.to_crs(epsg=2927)  # simulate a mismatched layer

layer_a_aligned, layer_b_aligned = ensure_same_crs(layer_a, layer_b, target_epsg=32610)
print("Layer A CRS:", layer_a_aligned.crs.to_epsg())
print("Layer B CRS:", layer_b_aligned.crs.to_epsg())

---
## Section 4 — Attribute Table Operations

In ArcGIS Pro, attribute table edits happen in the GUI: add field, calculate field, select by attributes. In code, the same operations are reproducible, batch-able, and auditable.

This section covers the four most common attribute workflows:
1. **Inspect** — what fields exist, what are the types, what does the data look like?
2. **Add and rename** — create derived columns, fix bad field names
3. **Filter** — select features by attribute value
4. **Compute** — run calculations across entire columns without a loop

We'll use Washington county data with population and area attributes — a dataset you'll recognize from your ArcGIS Pro work.

### 4a — Attribute operations in Python

In [ ]:
# [Python] Attribute table operations — same logic as ArcGIS 'Calculate Field' and 'Select by Attribute'

import geopandas as gpd
import pandas as pd
from geodatasets import get_path

# Load US states — we'll filter to Washington and work with its attributes
states = gpd.read_file(get_path('naturalearth.land'))

# Use a synthetic county-like dataset to keep the example reproducible
# In your workflow: counties = gpd.read_file('data/wa_counties.shp')
wa_counties = gpd.GeoDataFrame({
    'county':    ['King', 'Pierce', 'Snohomish', 'Spokane', 'Clark', 'Thurston', 'Kitsap', 'Yakima'],
    'pop_2020':  [2269675, 921130, 827957, 522798, 503320, 290536, 271473, 253340],
    'area_sqmi': [2307,    1676,   2090,   1764,   656,    727,    396,    4296],
    'region':    ['West', 'West', 'West', 'East', 'West', 'West', 'West', 'East'],
    'geometry':  gpd.points_from_xy(  # simplified — using centroids, not real polygons
        [-122.2, -122.4, -122.0, -117.4, -122.5, -122.9, -122.6, -120.5],
        [  47.5,   47.0,   47.9,   47.6,   45.7,   47.1,   47.7,   46.6]
    )
}, crs='EPSG:4326')

# ── 1. Inspect ──────────────────────────────────────────────────────────────
print("Shape:", wa_counties.shape)
print("\nColumn dtypes:")
print(wa_counties.dtypes)
print("\nFirst 3 rows:")
print(wa_counties[['county', 'pop_2020', 'area_sqmi', 'region']].head(3))
print("\nDescriptive stats:")
print(wa_counties[['pop_2020', 'area_sqmi']].describe().round(0))

# ── 2. Add derived columns ─────────────────────────────────────────────────
# Population density — vectorized, no loop needed
wa_counties['pop_density'] = (wa_counties['pop_2020'] / wa_counties['area_sqmi']).round(1)

# Classify density — using pd.cut() for binning (like Classify in ArcGIS Symbology)
wa_counties['density_class'] = pd.cut(
    wa_counties['pop_density'],
    bins=[0, 50, 200, 500, float('inf')],
    labels=['Rural', 'Suburban', 'Urban', 'Metro']
)

# Flag East WA counties
wa_counties['east_wa'] = wa_counties['region'] == 'East'

print("\nWith derived columns:")
print(wa_counties[['county', 'pop_density', 'density_class', 'east_wa']])

# ── 3. Filter (Select by Attribute equivalent) ─────────────────────────────
urban_or_metro = wa_counties[wa_counties['density_class'].isin(['Urban', 'Metro'])]
print(f"\nUrban/Metro counties: {len(urban_or_metro)}")
print(urban_or_metro[['county', 'pop_density', 'density_class']])

east_rural = wa_counties[(wa_counties['region'] == 'East') & (wa_counties['pop_density'] < 50)]
print(f"\nEast WA rural counties: {len(east_rural)}")
print(east_rural[['county', 'pop_density']])

### 4b — Attribute operations in R

In [ ]:
# [R] Attribute table operations with sf + dplyr
# dplyr verbs (mutate, filter, select, arrange) work directly on sf objects

library(sf)
library(dplyr)

# Build the same county dataset
wa_counties <- data.frame(
    county    = c('King', 'Pierce', 'Snohomish', 'Spokane', 'Clark', 'Thurston', 'Kitsap', 'Yakima'),
    pop_2020  = c(2269675, 921130, 827957, 522798, 503320, 290536, 271473, 253340),
    area_sqmi = c(2307, 1676, 2090, 1764, 656, 727, 396, 4296),
    region    = c('West', 'West', 'West', 'East', 'West', 'West', 'West', 'East'),
    lon       = c(-122.2, -122.4, -122.0, -117.4, -122.5, -122.9, -122.6, -120.5),
    lat       = c(  47.5,   47.0,   47.9,   47.6,   45.7,   47.1,   47.7,   46.6)
)
wa_counties_sf <- st_as_sf(wa_counties, coords = c('lon', 'lat'), crs = 4326)

# ── 1. Inspect ─────────────────────────────────────────────────────────────
cat("Dimensions:", nrow(wa_counties_sf), "rows x", ncol(wa_counties_sf), "cols\n")
cat("Column names:", paste(names(wa_counties_sf), collapse = ', '), "\n")
cat("\nColumn types:\n")
str(st_drop_geometry(wa_counties_sf))  # st_drop_geometry for non-spatial inspection

# ── 2. Add derived columns with dplyr::mutate() ─────────────────────────────
wa_counties_sf <- wa_counties_sf |>
    mutate(
        pop_density   = round(pop_2020 / area_sqmi, 1),
        density_class = case_when(
            pop_density < 50  ~ 'Rural',
            pop_density < 200 ~ 'Suburban',
            pop_density < 500 ~ 'Urban',
            TRUE              ~ 'Metro'
        ),
        east_wa = region == 'East'
    )

cat("\nWith derived columns:\n")
print(st_drop_geometry(wa_counties_sf) |> select(county, pop_density, density_class, east_wa))

# ── 3. Filter with dplyr::filter() ──────────────────────────────────────────
urban_or_metro <- wa_counties_sf |>
    filter(density_class %in% c('Urban', 'Metro'))
cat("\nUrban/Metro counties:", nrow(urban_or_metro), "\n")
print(st_drop_geometry(urban_or_metro) |> select(county, pop_density, density_class))

east_rural <- wa_counties_sf |>
    filter(region == 'East', pop_density < 50)
cat("\nEast WA rural counties:", nrow(east_rural), "\n")
print(st_drop_geometry(east_rural) |> select(county, pop_density))

# ── 4. Sort, select columns, and summarise ─────────────────────────────────
summary_by_region <- wa_counties_sf |>
    st_drop_geometry() |>
    group_by(region) |>
    summarise(
        n_counties   = n(),
        total_pop    = sum(pop_2020),
        avg_density  = round(mean(pop_density), 1)
    )
cat("\nSummary by region:\n")
print(summary_by_region)

---
## Section 5 — Visualizing Vector Data and Exporting a Clean GeoPackage

Quick visualization is a quality-control step, not just a presentation step. If your map looks wrong, your analysis is probably wrong. We'll produce a static map, then export a clean, documented GeoPackage as the module deliverable.

> **Python stack for vector visualization:**  
> - `geopandas.plot()` — quick static maps, built on matplotlib  
> - `matplotlib` — full control over figure layout, legends, insets  
> - `folium` — interactive Leaflet maps (web-ready HTML output)  
>
> **R stack:**  
> - `plot()` on an sf object — rapid inspection  
> - `ggplot2 + geom_sf()` — publication-quality static maps  
> - `tmap` — thematic maps with easy symbology control  
> - `leaflet` — interactive web maps from R

### 5a — Static vector map in Python

In [ ]:
# [Python] Quick static choropleth — population density by county

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Rebuild county GeoDataFrame with derived columns
wa_counties = gpd.GeoDataFrame({
    'county':    ['King', 'Pierce', 'Snohomish', 'Spokane', 'Clark', 'Thurston', 'Kitsap', 'Yakima'],
    'pop_2020':  [2269675, 921130, 827957, 522798, 503320, 290536, 271473, 253340],
    'area_sqmi': [2307,    1676,   2090,   1764,   656,    727,    396,    4296],
    'region':    ['West', 'West', 'West', 'East', 'West', 'West', 'West', 'East'],
    'geometry':  gpd.points_from_xy(
        [-122.2, -122.4, -122.0, -117.4, -122.5, -122.9, -122.6, -120.5],
        [  47.5,   47.0,   47.9,   47.6,   45.7,   47.1,   47.7,   46.6]
    )
}, crs='EPSG:4326')
wa_counties['pop_density'] = (wa_counties['pop_2020'] / wa_counties['area_sqmi']).round(1)

# ── Plot — proportional symbol map (point size = pop_density) ─────────────────
fig, ax = plt.subplots(figsize=(9, 6))

# Color points by region
colors = wa_counties['region'].map({'West': '#4b2e83', 'East': '#b7a57a'})

# Proportional symbol — scale marker size to pop_density
ax.scatter(
    wa_counties.geometry.x,
    wa_counties.geometry.y,
    s=wa_counties['pop_density'] / 3,   # scale down for display
    c=colors,
    alpha=0.8,
    edgecolors='white',
    linewidths=0.8
)

# Label each point
for _, row in wa_counties.iterrows():
    ax.annotate(
        f"{row['county']}\n{row['pop_density']:.0f}/mi²",
        xy=(row.geometry.x, row.geometry.y),
        xytext=(5, 5), textcoords='offset points',
        fontsize=7.5, color='#333333'
    )

# Legend
legend_handles = [
    mpatches.Patch(color='#4b2e83', label='West WA'),
    mpatches.Patch(color='#b7a57a', label='East WA')
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9)

ax.set_title('Washington Counties — Population Density (persons/sq mi)', fontsize=12, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('output/wa_density_map.png', dpi=150, bbox_inches='tight')
plt.show()
print("Map saved to output/wa_density_map.png")

### 5b — Static vector map in R with ggplot2 + geom_sf

In [ ]:
# [R] ggplot2 vector map — population density
# ggplot2 + geom_sf() is the workhorse for publication-quality static spatial plots in R

library(sf)
library(ggplot2)
library(dplyr)

# Rebuild county sf
wa_counties_sf <- st_as_sf(
    data.frame(
        county    = c('King', 'Pierce', 'Snohomish', 'Spokane', 'Clark', 'Thurston', 'Kitsap', 'Yakima'),
        pop_2020  = c(2269675, 921130, 827957, 522798, 503320, 290536, 271473, 253340),
        area_sqmi = c(2307, 1676, 2090, 1764, 656, 727, 396, 4296),
        region    = c('West', 'West', 'West', 'East', 'West', 'West', 'West', 'East'),
        lon       = c(-122.2, -122.4, -122.0, -117.4, -122.5, -122.9, -122.6, -120.5),
        lat       = c(  47.5,   47.0,   47.9,   47.6,   45.7,   47.1,   47.7,   46.6)
    ),
    coords = c('lon', 'lat'), crs = 4326
) |>
    mutate(pop_density = round(pop_2020 / area_sqmi, 1))

# ── ggplot2 proportional symbol map ──────────────────────────────────────────
ggplot(wa_counties_sf) +
    geom_sf(
        aes(color = region, size = pop_density),
        alpha = 0.85
    ) +
    geom_sf_label(
        aes(label = paste0(county, '\n', pop_density, '/mi²')),
        size = 2.6, nudge_y = 0.08, label.padding = unit(0.15, 'lines')
    ) +
    scale_color_manual(
        values = c('West' = '#4b2e83', 'East' = '#b7a57a'),
        name = 'Region'
    ) +
    scale_size_continuous(
        name = 'Density\n(persons/mi²)',
        range = c(3, 12)
    ) +
    labs(
        title    = 'Washington Counties — Population Density',
        subtitle = 'persons per square mile',
        caption  = 'Source: 2020 US Census'
    ) +
    theme_minimal(base_size = 11) +
    theme(
        plot.title    = element_text(face = 'bold', color = '#4b2e83'),
        legend.position = 'right'
    )

ggsave('output/wa_density_map_r.png', width = 9, height = 6, dpi = 150)
cat("Map saved to output/wa_density_map_r.png\n")

### 5c — Export a clean, documented GeoPackage

In [ ]:
# [Python] Export a clean, multi-layer GeoPackage — the preferred delivery format

import geopandas as gpd
import pandas as pd
import os

os.makedirs('output', exist_ok=True)

# Rebuild our county data
wa_counties = gpd.GeoDataFrame({
    'county':       ['King', 'Pierce', 'Snohomish', 'Spokane', 'Clark', 'Thurston', 'Kitsap', 'Yakima'],
    'pop_2020':     [2269675, 921130, 827957, 522798, 503320, 290536, 271473, 253340],
    'area_sqmi':    [2307,    1676,   2090,   1764,   656,    727,    396,    4296],
    'region':       ['West', 'West', 'West', 'East', 'West', 'West', 'West', 'East'],
    'geometry':  gpd.points_from_xy(
        [-122.2, -122.4, -122.0, -117.4, -122.5, -122.9, -122.6, -120.5],
        [  47.5,   47.0,   47.9,   47.6,   45.7,   47.1,   47.7,   46.6]
    )
}, crs='EPSG:4326')

wa_counties['pop_density'] = (wa_counties['pop_2020'] / wa_counties['area_sqmi']).round(1)
wa_counties['density_class'] = pd.cut(
    wa_counties['pop_density'],
    bins=[0, 50, 200, 500, float('inf')],
    labels=['Rural', 'Suburban', 'Urban', 'Metro']
).astype(str)  # cast to string for GPKG compatibility

# Build a stations layer for the second GPKG layer
stations = gpd.GeoDataFrame({
    'station_id': ['S01', 'S02', 'S03', 'S04'],
    'name':       ['Sea-Tac', 'Olympia', 'Bellingham', 'Spokane'],
    'precip_cm':  [96.0, 123.7, 95.3, 43.0],
    'geometry':   gpd.points_from_xy(
        [-122.3088, -122.9007, -122.5403, -117.4260],
        [  47.4444,   47.0379,   48.7519,   47.6588]
    )
}, crs='EPSG:4326')

gpkg_path = 'output/module3_wa_data.gpkg'

# Write first layer — creates the file
wa_counties.to_file(gpkg_path, driver='GPKG', layer='wa_counties')

# Append second layer — use mode='a'
stations.to_file(gpkg_path, driver='GPKG', layer='wx_stations', mode='a')

print(f"GeoPackage written: {gpkg_path}")

# Verify: list layers in the written GPKG
import fiona
layers = fiona.listlayers(gpkg_path)
print(f"Layers in GeoPackage: {layers}")

# Verify: read back and confirm
counties_check = gpd.read_file(gpkg_path, layer='wa_counties')
print(f"Counties layer: {len(counties_check)} features, CRS: {counties_check.crs.to_epsg()}")

---
## Section 6 — ArcPy and the ArcGIS API for Python: Vector Data Access

Everything above works in any Python or R environment. This section covers how the same vector concepts map to the ESRI ecosystem — because your production data almost certainly lives in a File Geodatabase or an ArcGIS feature service.

**When to use ArcPy vs. GeoPandas:**

| Scenario | Use |
|---|---|
| Data lives in a File GDB with domains/subtypes | `arcpy` — full schema awareness |
| Need to run ArcGIS geoprocessing tools | `arcpy.analysis`, `arcpy.management` |
| Data is Shapefile, GeoJSON, or GeoPackage | `geopandas` — faster for open formats |
| Connecting to ArcGIS Online or Portal feature services | `arcgis` API for Python |
| Mixed pipeline (ESRI tools → open analysis → back to ESRI) | Both, with GeoPackage as the hand-off format |

> **Environment note:** ArcPy only runs in the ArcGIS Pro Python environment (or a clone of it).  
> These cells will not run in a standard Jupyter environment — they are included here for reference and lab use inside ArcGIS Pro's built-in notebook.

In [ ]:
# [Python / ArcPy] Reading vector data from a File Geodatabase
# Run this inside ArcGIS Pro's built-in Notebook or in a cloned arcgispro-py3 environment

import arcpy
import os

# ── Set workspace to your File GDB ───────────────────────────────────────────
# arcpy.env.workspace = r'C:\Projects\WA_GIS\data\WA_Counties.gdb'

# ── List all feature classes in the GDB ─────────────────────────────────────
# fcs = arcpy.ListFeatureClasses()
# print("Feature classes:", fcs)

# ── Describe a feature class — geometry type, CRS, field info ────────────────
# desc = arcpy.Describe('counties')
# print("Geometry type:", desc.shapeType)
# print("Spatial reference:", desc.spatialReference.name)
# print("WKID:", desc.spatialReference.factoryCode)

# ── Read into a Pandas/GeoPandas DataFrame via SearchCursor ──────────────────
# fields = ['COUNTY_NAME', 'POP2020', 'AREA_SQMI', 'SHAPE@AREA', 'SHAPE@XY']
# rows = []
# with arcpy.da.SearchCursor('counties', fields) as cursor:
#     for row in cursor:
#         rows.append({
#             'county':     row[0],
#             'pop_2020':   row[1],
#             'area_sqmi':  row[2],
#             'shape_area': row[3],
#             'centroid_x': row[4][0],
#             'centroid_y': row[4][1]
#         })
# import pandas as pd
# df = pd.DataFrame(rows)
# print(df.head())

# ── Better: convert entire feature class to GeoDataFrame using arcgis package ──
# from arcgis.features import GeoAccessor
# import pandas as pd
# sdf = pd.DataFrame.spatial.from_featureclass('counties')
# print(type(sdf))        # a Spatially Enabled DataFrame
# print(sdf.columns)

# ── Run a geoprocessing tool on the feature class ───────────────────────────
# arcpy.analysis.Buffer(
#     in_features = 'counties',
#     out_feature_class = 'counties_5mi_buffer',
#     buffer_distance_or_field = '5 Miles'
# )
# print("Buffer complete.")

print("ArcPy cell — uncomment lines and run inside ArcGIS Pro Notebook or cloned env.")
print("In standard Jupyter: import will fail with ModuleNotFoundError — that's expected.")

In [ ]:
# [Python] ArcGIS API for Python — accessing vector data from ArcGIS Online or Portal
# Requires: pip install arcgis (or included in ArcGIS Pro environment)

# from arcgis.gis import GIS
# from arcgis.features import FeatureLayer
# import geopandas as gpd

# ── Connect to ArcGIS Online (anonymous for public data) ─────────────────────
# gis = GIS()   # anonymous public access
# gis = GIS('https://www.arcgis.com', 'your_username', 'your_password')  # authenticated

# ── Search for a public feature layer ────────────────────────────────────────
# results = gis.content.search('Washington Counties owner:esri', item_type='Feature Layer')
# for item in results[:3]:
#     print(item.title, '|', item.url)

# ── Access a feature layer by URL ─────────────────────────────────────────────
# layer_url = 'https://services.arcgis.com/.../FeatureServer/0'
# fl = FeatureLayer(layer_url)

# ── Query to a Spatially Enabled DataFrame ────────────────────────────────────
# sdf = fl.query(where="STATE_NAME = 'Washington'", as_df=True)
# print(type(sdf))

# ── Convert to GeoDataFrame for open-source analysis ─────────────────────────
# gdf = gpd.GeoDataFrame.from_features(fl.query(where="1=1").to_geojson['features'])
# print(gdf.shape)

# ── Write analysis results back to AGOL as a new hosted layer ─────────────────
# result_item = gdf.spatial.to_featureclass(
#     location = gis.content,
#     title    = 'WA County Density Analysis'
# )

print("ArcGIS API cell — uncomment and run with valid credentials.")
print("See: https://developers.arcgis.com/python/ for full API docs.")

---
## Section 7 — Lab: The Monday Morning Data Request

**Scenario:** It's Monday morning. Your supervisor sends you a CSV of building permit counts by census tract (2023 Q4). The tract boundaries live in a File Geodatabase your agency maintains. The web team needs a GeoPackage by noon.

Work through the steps below with a partner. Each step mirrors what you'd do in production.

**Using the synthetic dataset** (no file download needed):
- Tract boundaries are the census tracts we'll create here
- Permit CSV is constructed inline

**Goal:** A single GeoPackage with:
1. A `tracts` layer — polygon boundaries with population, area, and permit data joined
2. A `permit_centroids` layer — point layer of permit locations
3. Both layers in EPSG:32610 (UTM Zone 10N)
4. A `pop_density` and `permits_per_1k` derived column

In [ ]:
# [Python] Lab — The Monday Morning Data Request
# Complete the workflow: CSV join → CRS check → derived columns → export GeoPackage

import geopandas as gpd
import pandas as pd
import os

os.makedirs('output', exist_ok=True)

# ── Step 1: Load the tract 'boundaries' ───────────────────────────────────────
# In production: tracts = gpd.read_file('data/census_tracts.gdb', layer='tracts_2020')
# For lab: we simulate tracts as simple bounding-box polygons
from shapely.geometry import box

tract_data = {
    'tract_id': ['530330001', '530330002', '530330003', '530330004', '530330005'],
    'pop_2020': [4821, 3204, 6710, 2198, 5543],
    'area_sqmi': [0.82, 1.14, 0.67, 2.31, 0.95],
    'geometry': [
        box(-122.35, 47.60, -122.30, 47.64),
        box(-122.30, 47.60, -122.25, 47.64),
        box(-122.25, 47.60, -122.20, 47.64),
        box(-122.35, 47.55, -122.25, 47.60),
        box(-122.25, 47.55, -122.15, 47.60)
    ]
}
tracts = gpd.GeoDataFrame(tract_data, crs='EPSG:4326')
print("Step 1 — Tracts loaded:", len(tracts), "features, CRS:", tracts.crs.to_epsg())

# ── Step 2: Load the permit CSV ───────────────────────────────────────────────
permits_df = pd.DataFrame({
    'tract_id':    ['530330001', '530330002', '530330003', '530330004', '530330005'],
    'permits_q4':  [34, 12, 67, 5, 41],
    'permit_value_k': [8240, 2100, 18500, 430, 9870]   # in thousands
})
print("Step 2 — Permits CSV:", len(permits_df), "rows")

# ── Step 3: Join permits to tracts on tract_id ────────────────────────────────
tracts_joined = tracts.merge(permits_df, on='tract_id', how='left')
print("Step 3 — After join:", tracts_joined.columns.tolist())

# ── Step 4: Add derived columns ───────────────────────────────────────────────
tracts_joined['pop_density']    = (tracts_joined['pop_2020'] / tracts_joined['area_sqmi']).round(1)
tracts_joined['permits_per_1k'] = (tracts_joined['permits_q4'] / tracts_joined['pop_2020'] * 1000).round(2)
print("Step 4 — Derived columns added")

# ── Step 5: Reproject to UTM Zone 10N for metric analysis ────────────────────
tracts_utm = tracts_joined.to_crs(epsg=32610)
print("Step 5 — Reprojected to EPSG:", tracts_utm.crs.to_epsg())

# ── Step 6: Create point layer from centroids ──────────────────────────────────
permit_centroids = tracts_utm.copy()
permit_centroids.geometry = tracts_utm.geometry.centroid
permit_centroids = permit_centroids[['tract_id', 'permits_q4', 'permit_value_k', 'geometry']]
print("Step 6 — Centroid layer created:", len(permit_centroids), "points")

# ── Step 7: Export multi-layer GeoPackage ─────────────────────────────────────
out_gpkg = 'output/monday_delivery.gpkg'
tracts_utm.to_file(out_gpkg, driver='GPKG', layer='tracts')
permit_centroids.to_file(out_gpkg, driver='GPKG', layer='permit_centroids', mode='a')

print(f"\n✅  GeoPackage exported: {out_gpkg}")

import fiona
print("Layers:", fiona.listlayers(out_gpkg))
print("\nFinal attribute table:")
print(tracts_utm[['tract_id', 'pop_2020', 'pop_density', 'permits_q4', 'permits_per_1k']].to_string(index=False))

In [ ]:
# [R] Lab — The Monday Morning Data Request (R version)

library(sf)
library(dplyr)

# ── Step 1: Load tracts ─────────────────────────────────────────────────────
# In production: tracts <- st_read('data/census_tracts.gdb', layer = 'tracts_2020', quiet = TRUE)

# Lab: construct polygons using st_bbox → st_as_sfc trick
make_box <- function(xmin, ymin, xmax, ymax) {
    st_polygon(list(rbind(
        c(xmin, ymin), c(xmax, ymin),
        c(xmax, ymax), c(xmin, ymax),
        c(xmin, ymin)
    )))
}

tracts <- st_sf(
    tract_id  = c('530330001', '530330002', '530330003', '530330004', '530330005'),
    pop_2020  = c(4821, 3204, 6710, 2198, 5543),
    area_sqmi = c(0.82, 1.14, 0.67, 2.31, 0.95),
    geometry  = st_sfc(
        make_box(-122.35, 47.60, -122.30, 47.64),
        make_box(-122.30, 47.60, -122.25, 47.64),
        make_box(-122.25, 47.60, -122.20, 47.64),
        make_box(-122.35, 47.55, -122.25, 47.60),
        make_box(-122.25, 47.55, -122.15, 47.60),
        crs = 4326
    )
)
cat("Step 1 — Tracts:", nrow(tracts), "features | CRS:", st_crs(tracts)$epsg, "\n")

# ── Step 2: Load permit CSV ──────────────────────────────────────────────────
permits_df <- data.frame(
    tract_id      = c('530330001', '530330002', '530330003', '530330004', '530330005'),
    permits_q4    = c(34, 12, 67, 5, 41),
    permit_value_k = c(8240, 2100, 18500, 430, 9870)
)
cat("Step 2 — Permits:", nrow(permits_df), "rows\n")

# ── Step 3: Join ─────────────────────────────────────────────────────────────
tracts_joined <- tracts |> left_join(permits_df, by = 'tract_id')
cat("Step 3 — After join:", paste(names(tracts_joined), collapse = ', '), "\n")

# ── Step 4: Derived columns ───────────────────────────────────────────────────
tracts_joined <- tracts_joined |>
    mutate(
        pop_density    = round(pop_2020 / area_sqmi, 1),
        permits_per_1k = round(permits_q4 / pop_2020 * 1000, 2)
    )
cat("Step 4 — Derived columns added\n")

# ── Step 5: Reproject to UTM Zone 10N ────────────────────────────────────────
tracts_utm <- st_transform(tracts_joined, crs = 32610)
cat("Step 5 — Reprojected to EPSG:", st_crs(tracts_utm)$epsg, "\n")

# ── Step 6: Centroid point layer ──────────────────────────────────────────────
permit_centroids <- tracts_utm |>
    select(tract_id, permits_q4, permit_value_k) |>
    st_centroid()
cat("Step 6 — Centroid layer:", nrow(permit_centroids), "points\n")

# ── Step 7: Export GeoPackage ─────────────────────────────────────────────────
out_gpkg <- 'output/monday_delivery_r.gpkg'
st_write(tracts_utm,       out_gpkg, layer = 'tracts',           delete_layer = TRUE, quiet = TRUE)
st_write(permit_centroids, out_gpkg, layer = 'permit_centroids', append = TRUE,       quiet = TRUE)

cat("\n✅  GeoPackage exported:", out_gpkg, "\n")
cat("Layers:", paste(st_layers(out_gpkg)$name, collapse = ', '), "\n")
cat("\nFinal table:\n")
print(st_drop_geometry(tracts_utm) |>
      select(tract_id, pop_2020, pop_density, permits_q4, permits_per_1k))

---
## Section 8 — Extended Application: Multi-Format Workflow Pipeline

**Scenario extension:** Automate the Monday workflow so it runs on any input CSV and tract GDB without code changes. The function takes file paths as arguments and returns a GeoPackage path.

This is the pattern that makes the Monday morning request take 30 seconds instead of 3 hours.

In [ ]:
# [Python] A reusable, parameterized vector workflow function

import geopandas as gpd
import pandas as pd
import fiona
import os
from pathlib import Path

def build_permit_deliverable(
    tracts_path:   str,
    permits_csv:   str,
    join_field:    str,
    output_gpkg:   str,
    target_epsg:   int = 32610,
    layer_name:    str = 'tracts',
    tract_layer:   str = None   # for GDB multi-layer sources
):
    """
    Generic function to join a permit CSV to census tract polygons
    and export a clean, reprojected GeoPackage.

    Parameters
    ----------
    tracts_path   : path to vector file (Shapefile, GeoJSON, GDB, GPKG)
    permits_csv   : path to CSV containing permit counts
    join_field    : field name present in both datasets to join on
    output_gpkg   : path to write the output GeoPackage
    target_epsg   : EPSG code for output CRS (default: UTM 10N)
    layer_name    : layer name for the output tracts layer
    tract_layer   : layer name if tracts_path is a multi-layer source

    Returns
    -------
    str : path to the written GeoPackage
    """
    print(f"[1/6] Loading tracts from: {tracts_path}")
    read_kwargs = {}
    if tract_layer:
        read_kwargs['layer'] = tract_layer
    tracts = gpd.read_file(tracts_path, **read_kwargs)
    print(f"      {len(tracts)} features | CRS: {tracts.crs.to_epsg()}")

    print(f"[2/6] Loading permits from: {permits_csv}")
    permits = pd.read_csv(permits_csv)
    print(f"      {len(permits)} rows | Columns: {list(permits.columns)}")

    print(f"[3/6] Joining on field: '{join_field}'")
    if join_field not in tracts.columns:
        raise ValueError(f"join_field '{join_field}' not in tracts columns: {list(tracts.columns)}")
    if join_field not in permits.columns:
        raise ValueError(f"join_field '{join_field}' not in permits columns: {list(permits.columns)}")
    joined = tracts.merge(permits, on=join_field, how='left')

    print(f"[4/6] Computing derived columns")
    if 'pop_2020' in joined.columns and 'area_sqmi' in joined.columns:
        joined['pop_density'] = (joined['pop_2020'] / joined['area_sqmi']).round(1)
    if 'permits_q4' in joined.columns and 'pop_2020' in joined.columns:
        joined['permits_per_1k'] = (joined['permits_q4'] / joined['pop_2020'] * 1000).round(2)

    print(f"[5/6] Reprojecting to EPSG:{target_epsg}")
    if joined.crs.to_epsg() != target_epsg:
        joined = joined.to_crs(epsg=target_epsg)
    centroids = joined.copy()
    centroids.geometry = joined.geometry.centroid
    centroids = centroids[['tract_id', 'permits_q4', 'geometry']]

    print(f"[6/6] Writing GeoPackage: {output_gpkg}")
    Path(output_gpkg).parent.mkdir(parents=True, exist_ok=True)
    joined.to_file(output_gpkg, driver='GPKG', layer=layer_name)
    centroids.to_file(output_gpkg, driver='GPKG', layer='permit_centroids', mode='a')

    layers = fiona.listlayers(output_gpkg)
    print(f"\n✅  Done. Layers: {layers}")
    return output_gpkg


# ── Demo: create minimal test files and run the function ──────────────────────
from shapely.geometry import box
import json, tempfile

# Write a minimal GeoJSON tracts file
test_tracts = gpd.GeoDataFrame({
    'tract_id': ['530330001', '530330002', '530330003'],
    'pop_2020':  [4821, 3204, 6710],
    'area_sqmi': [0.82, 1.14, 0.67],
    'geometry': [box(-122.35, 47.60, -122.30, 47.64),
                 box(-122.30, 47.60, -122.25, 47.64),
                 box(-122.25, 47.60, -122.20, 47.64)]
}, crs='EPSG:4326')
test_tracts.to_file('output/test_tracts.geojson', driver='GeoJSON')

# Write a minimal permits CSV
pd.DataFrame({
    'tract_id':   ['530330001', '530330002', '530330003'],
    'permits_q4': [34, 12, 67]
}).to_csv('output/test_permits.csv', index=False)

# Run the pipeline
build_permit_deliverable(
    tracts_path = 'output/test_tracts.geojson',
    permits_csv = 'output/test_permits.csv',
    join_field  = 'tract_id',
    output_gpkg = 'output/monday_automated.gpkg'
)

---

## 🔍 Module 3 — Self-Check Questions

Work through these without looking at the cells above. Then verify by running code.

**Format and I/O**
1. What are two reasons you'd prefer GeoPackage over Shapefile as your output format?
2. You receive a CSV with columns `x` and `y` in State Plane feet (EPSG:2927). Write the Python line to create a GeoDataFrame with the correct CRS.
3. How do you add a second layer to an existing GeoPackage in GeoPandas without overwriting the first layer?

**Geometry**
4. What does `Polygon.buffer(0)` do, and when would you use it?
5. What's the difference between a `Polygon` and a `MultiPolygon`? Give a real-world example of each.
6. You have a LineString with 3 vertices. How do you access the length in the native units of its CRS?

**CRS**
7. Explain in one sentence: what does `to_crs()` do that `set_crs()` does not?
8. You want to compute the area of Washington state counties in square kilometers. What CRS would you use? Why not EPSG:4326?
9. You load two layers: one in EPSG:4326, one in EPSG:2927. Before running `gpd.sjoin()`, what must you do?

**Attributes**
10. You have a GeoDataFrame with a `population` column and an `area_km2` column. Write the single line to add a `density` column without using a for loop.
11. In R with dplyr, what function is the equivalent of ArcGIS "Select by Attribute"?
12. What does `st_drop_geometry()` do, and why is it useful when summarising attribute data?

---
## ✅ Module 3 Homework — Deliverables

Submit the following in Canvas before the next session:

### Deliverable 1 — Format Round-Trip (25 pts)
Pick **one** of your own agency datasets (or use the county data from this notebook):  
- Load it from its native format  
- Run the full 6-point sanity check (geometry type, CRS, row count, bounding box, null geometries)  
- Add at least one derived column  
- Export as a GeoPackage with at least two layers (polygons + point centroids)  

Submit: a screenshot of your sanity-check output **and** a screenshot of the GeoPackage opened in QGIS or ArcGIS Pro showing both layers.

### Deliverable 2 — CRS Alignment (25 pts)
Find two publicly available datasets covering the same study area but in **different CRS**:  
- Document each CRS (EPSG, name, type, units)  
- Show the bounding boxes before and after reprojection  
- Compute one distance or area measurement in a projected CRS and explain your choice of projection  

Submit: the notebook cells with output, plus one sentence explaining why you chose the CRS you did.

### Deliverable 3 — Parameterized Function (50 pts)
Adapt the `build_permit_deliverable()` function (or write your own) to solve a real or realistic workflow problem:  
- The function must take at least **two** file paths as arguments  
- It must reproject both inputs to a shared CRS  
- It must add at least one derived column  
- It must export a GeoPackage with at least two layers  
- Commit the notebook and output GeoPackage to your GitHub repo

Submit: your GitHub repo link (same repo from Module 2). Tag the commit `module3-deliverable`.

---

**Stuck?** Post in Ed Discussion — tag `#module3`. Include your error message, the cell that failed, and your OS/environment. If you solved something tricky, share how — the whole cohort benefits.